# El asesor de diseño en varios casos reales

`doekit.recommend_design` responde la pregunta *"¿qué método de diseño experimental
es el mejor para mi caso?"*. No es un AutoML mágico: es un **asesor transparente** que
combina dos capas —

1. **Reglas** (metodología clásica: Montgomery; handbook NIST/SEMATECH; guías de JMP)
   acotan el *shortlist* de métodos plausibles según el objetivo, el nº/tipo de factores
   y el orden del modelo.
2. **Evaluación** rankea ese shortlist con las métricas de `doekit` (D/A/G-eficiencia,
   varianza de predicción, nº de corridas) según tus **prioridades**, usando una
   **media geométrica ponderada**: un eje catastrófico (p. ej. predicción ≈ 0) *hunde*
   el score, sin que otro eje lo compense.

**Salvedad central:** "el mejor" es un **trade-off multiobjetivo** (pocas corridas vs
precisión de los coeficientes vs predicción en la región). Por eso el asesor **muestra
la tabla de alternativas** y expone sus supuestos — no esconde la decisión.

Este cuaderno recorre ocho casos lado a lado.


In [1]:
import pandas as pd
import doekit as ed
from IPython.display import display

def mostrar(r, extra_caveats=False):
    print("RECOMENDADO:", r.method, " | escenario:", r.scenario)
    print(r.rationale)
    display(r.table)
    if extra_caveats:
        for c in r.caveats:
            print("  -", c)


## Caso 1 — Screening con presupuesto ajustado

Seis factores, solo 8 corridas disponibles: ¿cuáles influyen? El asesor debe elegir un
diseño de screening que quepa en el presupuesto.


In [2]:
r1 = ed.recommend_design("screening", factors=6, budget=8, seed=0)
mostrar(r1)

RECOMENDADO: Plackett-Burman  | escenario: {'goal': 'screening', 'n_factors': 6, 'budget': 8, 'model_order': 'linear', 'mixture': False, 'split_plot': False, 'constraints': {'mixture': False, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To identify influential factors with 6 factors with a budget of 8 runs and a 'linear' model, the best compromise under your priorities is Plackett-Burman (8 runs, D-efficiency 100.0%, G-efficiency 100.0%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Plackett-Burman,8,100.0,100.0,2.98,True,True
1,Definitive Screening,13,79.9,96.8,3.57,False,True


**Lectura:** gana **Plackett-Burman** (8 corridas, ortogonal). El Definitive Screening
existe pero no cabe en el presupuesto, así que queda listado pero descartado.


## Caso 2 — Superficie de respuesta (RSM), sin restricciones

Tres factores continuos, objetivo optimizar. Aquí compiten Box-Behnken, Central Composite,
Definitive Screening y un D-óptimo. Con prioridades equilibradas:


In [3]:
r2 = ed.recommend_design("optimization", factors=3, seed=0)
mostrar(r2)

RECOMENDADO: D-optimal  | escenario: {'goal': 'optimization', 'n_factors': 3, 'budget': None, 'model_order': 'quadratic', 'mixture': False, 'split_plot': False, 'constraints': {'mixture': False, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To model the response surface and locate the optimum with 3 factors and a 'quadratic' model, the best compromise under your priorities is D-optimal (13 runs, D-efficiency 46.1%, G-efficiency 88.3%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Box-Behnken,15,36.6,51.5,5.78,True,True
1,Central Composite,22,63.8,75.1,4.15,True,True
2,Definitive Screening,9,NaN,NaN,NaN,True,False
3,D-optimal,13,46.1,88.3,7.25,True,True


**Lectura:** el **D-óptimo** gana por economía de corridas y buena predicción; el DSD
sale como *no soporta el modelo* (no puede estimar el cuadrático completo con todas las
interacciones). Box-Behnken y Central Composite quedan a la vista como alternativas
clásicas robustas si prefieres una plantilla conocida.


## Caso 3 — El mismo caso, pero cambiando las prioridades

Aquí se ve el corazón del asesor: **"el mejor" depende de qué priorices**. Con el mismo
problema del Caso 2, movemos los pesos.


In [4]:
for etiqueta, prio in [("Priorizo POCAS CORRIDAS", {"runs":6, "precision":1, "prediction":1}),
                       ("Priorizo PRECISION",      {"runs":1, "precision":6, "prediction":1})]:
    r = ed.recommend_design("optimization", factors=3, priorities=prio, seed=0)
    print("%-26s -> %-18s (%d corridas)" % (etiqueta, r.method, r.design.n_runs))


Priorizo POCAS CORRIDAS    -> D-optimal          (13 corridas)
Priorizo PRECISION         -> Central Composite  (22 corridas)


**Lectura:** al priorizar pocas corridas gana un diseño chico (D-óptimo); al priorizar
precisión de los coeficientes gana el **Central Composite** (más corridas, mayor D-eficiencia).
Mismo problema, distinta respuesta — porque no hay un "mejor" absoluto.


## Caso 4 — Región restringida

Cuando parte del espacio experimental es inviable (una esquina prohibida, una relación
entre factores), los diseños-plantilla no aplican y el asesor fuerza el **diseño óptimo**
sobre candidatos factibles.


In [5]:
r4 = ed.recommend_design("optimization", factors=3, constraints=ed.Constraints(irregular=True), seed=0)
mostrar(r4, extra_caveats=True)

RECOMENDADO: D-optimal  | escenario: {'goal': 'optimization', 'n_factors': 3, 'budget': None, 'model_order': 'quadratic', 'mixture': False, 'split_plot': False, 'constraints': {'mixture': False, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': True, 'has_exclude': False}}
To model the response surface and locate the optimum with 3 factors and a 'quadratic' model, the best compromise under your priorities is D-optimal (13 runs, D-efficiency 46.1%, G-efficiency 88.3%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,D-optimal,13,46.1,88.3,7.25,True,True


  - Recommendation is conditional on model_order='quadratic' (and effect_size=1.0 for power).
  - "Best" is a multi-objective trade-off (runs vs precision vs prediction): adjust 'priorities' for your case.
  - After the first experimental wave, use propose_next_runs / augment_design to choose the next batch (sequential DoE) instead of restarting from scratch.


## Caso 5 — Presupuesto insuficiente

¿Qué pasa si el presupuesto no alcanza para ningún diseño que soporte el modelo? El asesor
**no falla en silencio**: lo señala como salvedad y sugiere el menor viable.


In [6]:
r5 = ed.recommend_design("optimization", factors=4, budget=6, seed=0)
mostrar(r5)
print("\nSalvedad principal:")
print(" ", r5.caveats[0])


RECOMENDADO: Definitive Screening  | escenario: {'goal': 'optimization', 'n_factors': 4, 'budget': 6, 'model_order': 'quadratic', 'mixture': False, 'split_plot': False, 'constraints': {'mixture': False, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To model the response surface and locate the optimum with 4 factors with a budget of 6 runs and a 'quadratic' model, the best compromise under your priorities is Definitive Screening (9 runs).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Box-Behnken,27,25.2,31.6,10.78,False,True
1,Central Composite,32,76.2,100.0,5.27,False,True
2,Definitive Screening,9,NaN,NaN,NaN,False,False
3,D-optimal,18,44.8,59.4,13.06,False,True



Salvedad principal:
  No catalog design fits the budget (6) and supports the model; recommending the smallest viable option (Definitive Screening, 9 runs). Increase the budget or reduce the model.


## Caso 6 — Factores categóricos

Los diseños RSM (Box-Behnken/CCD) asumen factores continuos. Con un factor categórico el
asesor evita esos métodos, prefiere factorial o D-óptimo, y **lo advierte**.


In [7]:
r6 = ed.recommend_design(
    "screening",
    factors=[ed.ContinuousFactor("x1", 0, 1), ed.CategoricalFactor("mat", ["A", "B", "C"])],
    seed=0)
mostrar(r6)
print("\nSalvedades relevantes:")
for c in r6.caveats:
    if "categor" in c.lower() or "mezcla" in c.lower():
        print(" -", c)


RECOMENDADO: D-optimal  | escenario: {'goal': 'screening', 'n_factors': 2, 'budget': None, 'model_order': 'linear', 'mixture': False, 'split_plot': False, 'constraints': {'mixture': False, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To identify influential factors with 2 factors and a 'linear' model, the best compromise under your priorities is D-optimal (7 runs, D-efficiency 41.6%, G-efficiency 85.7%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Full factorial,6,NaN,NaN,NaN,True,False
1,D-optimal,7,41.6,85.7,3.5,True,True



Salvedades relevantes:
 - Categorical factors present: RSM designs (Box-Behnken/CCD) assume continuous factors; prefer full factorial or D-optimal.


## Resumen de los casos


In [8]:
casos = [("1. Screening 6f, budget 8", r1),
         ("2. RSM 3f (balance)", r2),
         ("4. Region restringida", r4),
         ("5. Budget insuficiente 4f", r5),
         ("6. Categorico", r6)]
resumen = pd.DataFrame([{"caso": n, "recomendado": r.method,
                         "corridas": r.design.n_runs} for n, r in casos])
display(resumen)

,caso,recomendado,corridas
0,"1. Screening 6f, budget 8",Plackett-Burman,8
1,2. RSM 3f (balance),D-optimal,13
2,4. Region restringida,D-optimal,13
3,5. Budget insuficiente 4f,Definitive Screening,9
4,6. Categorico,D-optimal,7


## Caso 7 — Mezcla (suma constante)

Formulaciones: factores que suman 1. El advisor shortlistea simplex lattice/centroid
y modelos Scheffé (no un factorial en hipercubo).


In [9]:
r7 = ed.recommend_design(
    "optimization",
    factors=[ed.MixtureFactor("A"), ed.MixtureFactor("B"), ed.MixtureFactor("C")],
    mixture=True, seed=0,
)
mostrar(r7, extra_caveats=True)
# catálogo directo:
mix = ed.simplex_lattice(3, degree=2)
print("simplex_lattice:", mix.n_runs, "runs |", mix.metadata.get("kind"))


RECOMENDADO: Simplex lattice  | escenario: {'goal': 'optimization', 'n_factors': 3, 'budget': None, 'model_order': 'quadratic', 'mixture': True, 'split_plot': False, 'constraints': {'mixture': True, 'hard_to_change': [], 'split_plot': False, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To model a mixture response (Scheffé) on the simplex with 3 factors and a 'quadratic' model, the best compromise under your priorities is Simplex lattice (6 runs, D-efficiency 4.2%, G-efficiency 100.0%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Simplex lattice,6,4.2,100.0,3.80,True,True
1,Simplex centroid,7,3.9,92.8,3.49,True,True


  - Recommendation is conditional on model_order='quadratic' (and effect_size=1.0 for power).
  - "Best" is a multi-objective trade-off (runs vs precision vs prediction): adjust 'priorities' for your case.
  - After the first experimental wave, use propose_next_runs / augment_design to choose the next batch (sequential DoE) instead of restarting from scratch.
  - Mixture case: shortlist uses simplex lattice / centroid with Scheffé models; evaluate samples the simplex (not a hypercube).
simplex_lattice: 6 runs | SimplexLattice


## Caso 8 — Factores difíciles de cambiar (split-plot)

Cuando un factor (p. ej. temperatura de horno) no se puede randomizar corrida a corrida,
el advisor propone split-plot; el análisis usa `fit_mixed_model(..., groups="whole_plot_id")`.


In [10]:
r8 = ed.recommend_design(
    "screening",
    factors={"temp": (20, 80), "pH": (3, 9), "agit": (100, 400)},
    hard_to_change=["temp"], seed=0,
)
mostrar(r8, extra_caveats=True)
spd = ed.split_plot_design(
    whole_plot=[ed.ContinuousFactor("temp", 20, 80)],
    subplot=[ed.ContinuousFactor("pH", 3, 9),
             ed.ContinuousFactor("agit", 100, 400)],
    whole_plot_reps=2, seed=0,
)
print(spd.matrix[["temp", "pH", "agit", "whole_plot_id"]].head())
print("hard_to_change:", spd.metadata.get("hard_to_change"))


RECOMENDADO: D-optimal  | escenario: {'goal': 'screening', 'n_factors': 3, 'budget': None, 'model_order': 'linear', 'mixture': False, 'split_plot': True, 'constraints': {'mixture': False, 'hard_to_change': ['temp'], 'split_plot': True, 'run_cost': 1.0, 'irregular': False, 'has_exclude': False}}
To respect hard-to-change factors via a split-plot structure with 3 factors and a 'linear' model, the best compromise under your priorities is D-optimal (7 runs, D-efficiency 96.1%, G-efficiency 61.4%).


,method,runs,D_eff,G_eff,SPV_mean,in_budget,supports_model
0,Split-plot,16,100.0,100.0,2.00,True,True
1,D-optimal,7,96.1,61.4,2.19,True,True


  - Recommendation is conditional on model_order='linear' (and effect_size=1.0 for power).
  - "Best" is a multi-objective trade-off (runs vs precision vs prediction): adjust 'priorities' for your case.
  - After the first experimental wave, use propose_next_runs / augment_design to choose the next batch (sequential DoE) instead of restarting from scratch.
  - Split-plot case: analyse with fit_mixed_model(groups='whole_plot_id') (or blocks= for fixed plots). D-optimal alternative ignores HTC structure.
   temp  pH  agit  whole_plot_id
0    20   3   100              0
1    20   9   100              0
2    20   3   400              0
3    20   9   400              0
4    80   3   100              1
hard_to_change: ['temp']


## Puente: `ed.experiment` + export

Un call produce diseño + plan lab-ready; útil como handoff hacia el laboratorio o un agente.


In [11]:
exp = ed.experiment(goal="screening", factors=6, budget=12, seed=0)
print(exp.recommendation.method, "|", exp.design.n_runs, "runs")
display(exp.plan.head())
path = exp.export_csv("reports/08_asesor/runs.csv")
print("CSV:", path)
print("schema:", exp.to_dict()["schema"])


Plackett-Burman | 8 runs


,run_id,factor1,factor2,factor3,factor4,factor5,factor6,dummy1,y
0,1,1,1,1,1,1,1,1,<NA>
1,2,-1,1,-1,1,-1,1,-1,<NA>
2,3,1,-1,-1,1,1,-1,-1,<NA>
3,4,-1,-1,1,1,-1,-1,1,<NA>
4,5,1,1,1,-1,-1,-1,-1,<NA>


CSV: reports\08_asesor\runs.csv
schema: doekit.Experiment/1


## Conclusión

El asesor **acota con reglas y decide con métricas**, siempre exponiendo las alternativas y
las salvedades. Su valor no es dar "la" respuesta, sino hacer explícito el trade-off.

Desde **0.6**, mezcla (`mixture=` / `MixtureFactor` / `simplex_*`) y split-plot
(`hard_to_change=` / `split_plot_design`) entran en el shortlist del advisor — ya no
quedan "fuera del catálogo". El análisis mixed de whole-plots está en el notebook 09;
el bucle `ed.experiment(...)` (plan → ingest → next → export) es el contrato 0.7.

Esta misma lógica alimenta la sección *"¿Fue el diseño apropiado?"* de cada reporte.
CLI: `doekit recommend --factors 6 --budget 12 --export runs.csv`.
